# Multiple Linear Regression Project

In this notebook I complete each step of the multiple linear regression project using the diamonds dataset from Seaborn.

## Question 1 - Load the Dataset

First, I load the diamonds dataset into a DataFrame so it can be used throughout the project.

In [ ]:
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import SelectKBest, f_regression
import matplotlib.pyplot as plt

data = sns.load_dataset('diamonds')

## Question 2 - Display the First Five Rows

Next, I display the first five rows to make sure the dataset loaded correctly.

In [ ]:
print(data.head())

## Question 3 - Remove Unneeded Columns

The x, y, and z columns are not needed for this project, so I remove them before building the model.

In [ ]:
data = data.drop(columns=['x', 'y', 'z'])

## Question 4 - Display the Data Types

Here I use the info() method to check the data types for each column.

In [ ]:
data.info()

## Question 5 - Create Dummy Variables

Since regression models require numeric data, I convert the categorical columns into dummy variables.

In [ ]:
dummy_vars = pd.get_dummies(
    data[['cut', 'color', 'clarity']],
    drop_first=True
)

## Question 6 - Join the Dummy Variables

After creating the dummy variables, I remove the original categorical columns and join the new dummy columns to the DataFrame. I also display the updated information to verify the changes.

In [ ]:
dataDummies = data.drop(columns=['cut', 'color', 'clarity']).join(dummy_vars)

dataDummies.info()

## Question 7 - Rescale the Numeric Data

Next, I standardize the numeric predictor columns so they are on the same scale before creating the model.

In [ ]:
numeric_cols = dataDummies.drop(columns=['price']).select_dtypes(include='number').columns

scaler = StandardScaler()

dataDummies[numeric_cols] = scaler.fit_transform(dataDummies[numeric_cols])

print(dataDummies.head())

## Question 8 - Display the Correlation with Price

Here I display the correlation values to see which features are most closely related to the price.

In [ ]:
correlation = dataDummies.corr()['price'].sort_values(ascending=False)

print(correlation)

## Question 9 - Create the Training and Testing Data

Using the five features with the highest correlation to price, I split the data into training and testing datasets. I use a 70/30 split with a random state so the results stay consistent.

In [ ]:
top_5_features = correlation.drop('price').abs().nlargest(5).index

X = dataDummies[top_5_features]

y = dataDummies['price']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

## Question 10 - Create and Train the Model

Now I create a linear regression model and fit it using the training data.

In [ ]:
model = LinearRegression()

model.fit(X_train, y_train)

## Question 11 - Test Score

After training the model, I calculate the score using the testing dataset to see how well it performs on unseen data.

In [ ]:
test_score = model.score(X_test, y_test)

print("Test Score:", test_score)

## Question 12 - Training Score

I also calculate the score using the training dataset so I can compare it with the testing score.

In [ ]:
train_score = model.score(X_train, y_train)

print("Training Score:", train_score)

## Question 13 - Make Predictions

Next, I use the trained model to predict prices for the testing data.

In [ ]:
predictions = model.predict(X_test)

## Question 14 - Compare Actual and Predicted Prices

To compare the results, I create a DataFrame that includes the predictor values along with the actual and predicted prices.

In [ ]:
results_df = X_test.copy()

results_df['Actual Price'] = y_test

results_df['Predicted Price'] = predictions

print(results_df.head())

## Question 15 - Calculate the Residuals

The residuals are calculated by subtracting the predicted price from the actual price. This shows how far each prediction is from the true value.

In [ ]:
results_df['Residuals'] = (
    results_df['Actual Price']
    - results_df['Predicted Price']
)

print(results_df.head())

## Question 16 - Plot the Residuals

Here I create a KDE plot to visualize the distribution of the residual values.

In [ ]:
plt.figure(figsize=(8,5))

sns.kdeplot(results_df['Residuals'], fill=True)

plt.title("Residual Distribution")

plt.xlabel("Residual")

plt.show()

## Question 17 - Test Different Numbers of Features

In this step, I use SelectKBest inside a loop to test different numbers of features. I record both the training and testing scores for each model.

In [ ]:
X_all = dataDummies.drop('price', axis=1)

y_all = dataDummies['price']

X_train_all, X_test_all, y_train_all, y_test_all = train_test_split(
    X_all,
    y_all,
    test_size=0.30,
    random_state=42
)

train_scores = []

test_scores = []

num_features_list = range(1, len(X_all.columns) + 1)

for k in num_features_list:

    selector = SelectKBest(score_func=f_regression, k=k)

    X_train_k = selector.fit_transform(X_train_all, y_train_all)

    X_test_k = selector.transform(X_test_all)

    temp_model = LinearRegression()

    temp_model.fit(X_train_k, y_train_all)

    train_scores.append(
        temp_model.score(X_train_k, y_train_all)
    )

    test_scores.append(
        temp_model.score(X_test_k, y_test_all)
    )

## Question 18 - Plot the Scores

Finally, I plot the training and testing scores against the number of selected features to compare how the model performs as more features are added.

In [ ]:
plt.figure(figsize=(10,6))

sns.lineplot(
    x=list(num_features_list),
    y=train_scores,
    marker='o',
    label='Training Score'
)

sns.lineplot(
    x=list(num_features_list),
    y=test_scores,
    marker='s',
    label='Test Score'
)

plt.title("Training and Testing Scores by Number of Features")

plt.xlabel("Number of Features")

plt.ylabel("R² Score")

plt.grid(True)

plt.legend()

plt.show()